# Заняття 12. Експлуатація та підтримка мультиагентних систем

## Результати навчання

Після цього заняття ви:
1. **Вмієте** підключати LLM observability (LangSmith / Langfuse) до мультиагентної системи на LangGraph
2. **Знаєте** ключові функції LangSmith та Langfuse і вмієте обирати інструмент під задачу
3. **Вмієте** виносити промпти у Prompt Registry та керувати їх версіями
4. **Розумієте** стратегії оптимізації вартості: Prompt Caching (provider-level) та Semantic Caching (application-level)
5. **Вмієте** збирати user feedback та прив'язувати його до трейсів для побудови Data Flywheel

---

## 🔧 Налаштування середовища

Встановлюємо необхідні бібліотеки та налаштовуємо API ключі.

### Як отримати Langfuse ключі

1. Зареєструйтеся на **[us.cloud.langfuse.com](https://us.cloud.langfuse.com)** (безкоштовний tier доступний одразу, без credit card)
2. Після логіну створіть нову **Organization** (наприклад, `personal`) — це верхній рівень namespace
3. Усередині organization створіть **Project** (наприклад, `lecture-12`) — трейси з цього ноутбука потраплятимуть саме сюди
4. У сайдбарі проєкту: **Settings → API Keys → + Create new API keys**
5. Langfuse покаже пару ключів **одноразово** — скопіюйте обидва:
   - `Public Key` — починається з `pk-lf-...` → значення для `LANGFUSE_PUBLIC_KEY`
   - `Secret Key` — починається з `sk-lf-...` → значення для `LANGFUSE_SECRET_KEY`
6. Host вже захардкоджено у комірці нижче як `https://us.cloud.langfuse.com` — не плутайте з `cloud.langfuse.com` (EU регіон, інша база даних)

> ⚠️ `Secret Key` показується тільки один раз при створенні. Якщо втратили — створіть нову пару ключів і видаліть стару.

Наступна комірка запитає ці два ключі через `getpass` і збереже їх у environment variables на поточну notebook сесію.

In [1]:
# Install dependencies
!pip install -q langchain langchain-openai langgraph langsmith langfuse openai numpy ddgs

In [2]:
# API keys configuration
import os
from getpass import getpass

# --- LLM Providers ---
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("🔑 Enter your OpenAI API key: ")

# --- Langfuse (observability) ---
# Get keys at https://us.cloud.langfuse.com → Project Settings → API Keys
os.environ["LANGFUSE_BASE_URL"] = "https://us.cloud.langfuse.com"
if not os.getenv("LANGFUSE_PUBLIC_KEY"):
    os.environ["LANGFUSE_PUBLIC_KEY"] = getpass("🔑 Enter your LANGFUSE_PUBLIC_KEY (pk-lf-...): ")
if not os.getenv("LANGFUSE_SECRET_KEY"):
    os.environ["LANGFUSE_SECRET_KEY"] = getpass("🔑 Enter your LANGFUSE_SECRET_KEY (sk-lf-...): ")

🔑 Enter your OpenAI API key:  ········
🔑 Enter your LANGFUSE_PUBLIC_KEY (pk-lf-...):  ········
🔑 Enter your LANGFUSE_SECRET_KEY (sk-lf-...):  ········


---

## 1. Production day 2: як не втратити контроль над мультиагентною системою

### Чому стандартний logging не працює для агентів?

Уявіть: користувач задає питання вашій мультиагентній системі. Один запит легко породжує цілий ланцюжок LLM-викликів — reasoning, tool calling, observation, reflection, делегування між агентами. 

Тут класичому `logging.info("response sent")` буде складно описати те, **чому** система прийняла конкретне рішення.

### Виклики MAS:

**1. Каскадні збої (Cascading Failures)**  
Агент A передав неправильний JSON агенту B → Агент B "галюцинує" відповідь → Критичне рішення прийнято на основі сміття. Без трейсингу ви не знаєте, де саме зламалось.

**2. Непередбачуване споживання токенів**  
Агент може зациклитися на виклику інструменту (retry loop). Один "поганий" запит здатен спожити токенів на суму, що значно перевищує очікувану вартість одного звернення. Без моніторингу ви дізнаєтесь про це з рахунку в кінці місяця.

**3. Складність дебагінгу**  
У системі з 3+ агентами та conditional routing — десятки можливих шляхів виконання. Помилка може виникнути в одному агенті, а проявитися в іншому.

Експлуатація мультиагентної системи — це не просто "додайте логи". Це інфраструктура для розуміння, **чому** ваша система прийняла конкретне рішення, **скільки** це коштувало, і **як** це покращити.

---

## 2. Observability та трейсинг: LangSmith та Langfuse

### 2.1. Що таке LLM Tracing?

**Трейсинг** — це структуроване записування повного шляху кожного запиту через систему. Три ключові концепції:

- **Trace** — повний шлях одного запиту користувача (від запитання до фінальної відповіді)
- **Span** (або **Observation**) — одна атомарна операція всередині трейсу (один LLM-виклик, один tool call, одна ретрівал-операція)
- **Session** — група трейсів, пов'язаних з одним користувачем або діалогом

Кожен спан зберігає:
- **Input** (prompt) та **Output** (response)
- Модель та її параметри (`temperature`, `max_tokens`)
- Кількість токенів (input/output) та **вартість**
- **Latency** (час виконання)
- Метадані (`user_id`, `session_id`, `tags`)

**Trace Tree: мультиагентний workflow на запит "Порівняй iPhone 16 та Galaxy S25"** _(ілюстративні значення)_

```
📋 Trace — total: 14.8s, $0.019
├── 🔀 Router Agent — gpt-5.4-nano             0.3s   $0.0002
├── 🔍 Research Agent #1 — gpt-5.4-nano        2.1s   $0.0003
│   ├── 🛠 Web Search Tool                     0.8s
│   └── 📝 Summarizer — gpt-5.4-nano           0.9s   $0.0002
├── 🔍 Research Agent #2 — gpt-5.4-nano        2.0s   $0.0003
│   ├── 🛠 Web Search Tool                     0.7s
│   └── 📝 Summarizer — gpt-5.4-nano           0.8s   $0.0002
├── ✍️  Writer — gpt-5.4-mini                  3.2s   $0.0018
└── 🎯 Critic — gpt-5.4                        3.8s   $0.0054
```

Кожен рядок — окремий **span**: або виклик LLM (з моделлю, latency, cost), або tool call. Вкладеність відповідає ієрархії викликів у LangGraph. Приклад того, як подібний trace виглядає у Langfuse UI — [Langfuse Tracing Overview](https://langfuse.com/docs/observability/overview).

### 2.2. Інструменти: LangSmith vs Langfuse

#### LangSmith — платформа від LangChain

| Категорія | Функціональність |
|-----------|-----------------|
| **Трейсинг** | Нативна інтеграція з LangChain/LangGraph — вмикається однією env var (`LANGSMITH_TRACING=true`). Structured timeline. SDK: Python, TypeScript, Go, Java. OpenTelemetry |
| **Monitoring** | Real-time дашборди (latency, cost, errors). Alerts при перевищенні порогів. Online evals на production трейсах |
| **Insights Agent** | AI-агент, що кластеризує production трейси за патернами використання ("що питають?") або типами збоїв ("як ламається?") — [анонс у блозі LangChain](https://blog.langchain.com/insights-agent-multiturn-evals-langsmith/). Доступний на Plus/Enterprise |
| **Evaluation** | Annotation Queues (human review), Multi-turn Evals (оцінка threads), LLM-as-a-Judge, Pairwise comparison, Polly (AI-асистент для аналізу) |
| **Prompt Hub** | Версіонування з labels (`dev`/`staging`/`prod`), Playground, зв'язок версій з метриками per chain step |
| **Deployment** | Cloud, BYOC, Self-hosted (Enterprise) |

#### Langfuse — open-source (MIT); у [2026 році приєдналась до ClickHouse](https://clickhouse.com/blog/clickhouse-acquires-langfuse-open-source-llm-observability)

| Категорія | Функціональність |
|-----------|-----------------|
| **Трейсинг** | Nested tracing необмеженої глибини. Sessions для multi-turn. User tracking. Environment separation. `@observe()` або `CallbackHandler`. OpenTelemetry backend |
| **Dashboards** | Curated + Custom (конструктор віджетів). Multi-level aggregations (trace → user → session). Фільтрація за metadata, scores, tool calls |
| **Evaluation** | Scores (Numeric/Categorical/Boolean/Text). LLM-as-a-Judge (online, **step-wise** на окремих operations). Annotation Queues. Score Analytics |
| **Datasets** | Test sets з production трейсів. Experiments з auto-evaluation. Versioned datasets. In-UI comparison tables |
| **Prompt Mgmt** | Version control + labels + Playground + Prompt folders. Кешування клієнт/сервер. Зв'язок промптів з трейсами |
| **Deployment** | Cloud (EU/US), Self-hosted (Docker/K8s). Є безплатний tier з місячним лімітом — актуальні обсяги див. на сайті |

#### Коли що обирати?

| Критерій | LangSmith | Langfuse |
|----------|-----------|----------|
| **Екосистема** | LangChain/LangGraph-first | Фреймворк-агностик |
| **Setup** | 1 env var для LangChain | 3 env vars + callback |
| **Self-hosting** | Enterprise only | Безкоштовно (MIT) |
| **Унікальна фіча** | Insights Agent (AI-аналіз) | Step-wise LLM-as-Judge |
| **Ціна** | Free → pay per volume | Free 50k, self-hosted $0 |

> 🔑 **Практична порада:** Для курсового проєкту — **Langfuse Cloud** (free tier, простий setup). Для production з LangGraph — **LangSmith** (нативна інтеграція, Insights Agent).

### 2.3. Мультиагентна система з трейсингом Langfuse

Побудуємо мультиагентну систему **Researcher → Writer** на LangGraph. Researcher має реальний tool `web_search`, сам вирішує, скільки пошукових запитів зробити, і передає знахідки Writer-у. Langfuse трейсить кожен крок через `CallbackHandler`, який ми передаємо у `RunnableConfig` — самих агентів торкатися не треба.

**Model routing** — кожен агент використовує модель відповідного tier:
- 🟢 **Researcher** → `gpt-5.4-mini` (tool calling, кілька ітерацій)
- 🔴 **Writer** → `gpt-5.4` (якісна синтез фінального тексту)

In [3]:
from langgraph.graph import MessagesState

In [4]:
# Step 2: Model identifiers — picked per agent role
# We use the provider-prefixed string form that `create_agent` accepts directly.

LLM_FAST = "openai:gpt-5.4-mini"       # tool-calling researcher
LLM_POWERFUL = "openai:gpt-5.4"        # writer — final synthesis

In [12]:
# Step 3: Real tool + agents via create_agent
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from ddgs import DDGS

@tool
def web_search(query: str) -> str:
    """Search the web via DuckDuckGo. Returns up to 5 results as a bullet list with title, snippet, and URL."""
    results = DDGS().text(query, max_results=5)
    if not results:
        return "No results found."
    return "\n".join(
        f"- {r['title']}: {r['body']} ({r['href']})" for r in results
    )

researcher_agent = create_agent(
    model=LLM_FAST,
    tools=[web_search],
    system_prompt=(
        "You are a research analyst. For the given topic, run 2-3 focused "
        "web_search calls, then summarize findings as 5-7 bullet points with "
        "sources (URLs). Be concrete and factual."
    ),
    name="researcher",
)

writer_agent = create_agent(
    model=LLM_POWERFUL,
    tools=[],
    system_prompt=(
        "You are a technical writer. The prior messages contain a user topic and "
        "a research summary produced by a researcher agent. Using that context, "
        "write a concise, well-structured article (250-350 words) about the topic. "
        "Cite sources inline."
    ),
    name="writer",
)

In [13]:
# Step 4: Wire compiled agents as nodes directly — no wrapper functions.
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(MessagesState)
workflow.add_node("researcher", researcher_agent)
workflow.add_node("writer", writer_agent)

workflow.add_edge(START, "researcher")
workflow.add_edge("researcher", "writer")
workflow.add_edge("writer", END)

app = workflow.compile()

print("✅ Graph compiled")
print("   Flow: researcher → writer → END")

✅ Graph compiled
   Flow: researcher → writer → END


In [14]:
# Step 5: Run the system — one Langfuse trace, grouped into a session
import uuid
from langchain_core.runnables import RunnableConfig
from langfuse import observe, propagate_attributes, get_client
from langfuse.langchain import CallbackHandler

langfuse = get_client()
langfuse_handler = CallbackHandler()  # reads LANGFUSE_* env vars

SESSION_ID = f"lecture-12-demo-{uuid.uuid4().hex[:8]}"

@observe(name="article-writer-run")
def run_mas(topic: str) -> tuple[dict, str]:
    # Capture the current trace id so we can attach user feedback later
    trace_id = langfuse.get_current_trace_id()
    with propagate_attributes(
        session_id=SESSION_ID,
        user_id="student_demo",
        tags=["demo", "multi-agent", "article-writer"],
        metadata={"lecture": "12", "experiment": "baseline_v1"},
    ):
        result = app.invoke(
            {"messages": [HumanMessage(content=(
                f"Topic: {topic}\n\n"
                "Step 1: research the topic using web_search.\n"
                "Step 2: write a 250-350 word article based on the research."
            ))]},
            config=RunnableConfig(
                callbacks=[langfuse_handler],
                recursion_limit=25,
            ),
        )
        return result, trace_id

print(f"🚀 Running multi-agent article writer (session: {SESSION_ID})...")
print("=" * 60)

result, last_trace_id = run_mas("Compare LangSmith and Langfuse as LLM observability platforms")

# Ensure pending spans are sent before inspecting the UI
langfuse.flush()

print("=" * 60)
print(f"\n📝 Final article preview (first 500 chars):\n")
print(result["messages"][-1].content[:500] + "...")
print(f"\n🆔 trace_id: {last_trace_id}  (used in Section 5 to attach feedback)")

🚀 Running multi-agent article writer (session: lecture-12-demo-f92da15a)...

📝 Final article preview (first 500 chars):

LangSmith and Langfuse both help teams observe, debug, and improve LLM applications, but they target slightly different buyers. LangSmith is LangChain’s integrated platform for tracing, monitoring, evaluation, and deployment workflows, with especially tight support for LangChain and LangGraph applications (LangSmith docs: https://docs.langchain.com/langsmith/home). If your stack is already built around LangChain, that native fit is its biggest advantage.

Langfuse takes a more open, framework-ag...

🆔 trace_id: 7a69dc425c3bc230c2d65f4d0bf63027  (used in Section 5 to attach feedback)


### 2.4. Що ми бачимо в Langfuse UI

Відкрийте [Langfuse US Cloud](https://us.cloud.langfuse.com) → Tracing → Traces. Знайдіть свіжий trace за тегом `multi-agent` або за `user_id:student_demo`.

**Trace Tree (Details View)** покаже приблизно таке:

```
📋 LangGraph run — total: ~12s, ~$0.02
├── 🔍 researcher (agent)                       5.8s
│   ├── 🤖 ChatOpenAI [gpt-5.4-mini]    decide search #1
│   ├── 🛠 web_search("langsmith observability") 0.9s
│   ├── 🤖 ChatOpenAI [gpt-5.4-mini]    decide search #2
│   ├── 🛠 web_search("langfuse features")        0.8s
│   └── 🤖 ChatOpenAI [gpt-5.4-mini]    write summary
└── ✍️  writer (agent)                           3.9s
    └── 🤖 ChatOpenAI [gpt-5.4]         generate article
```

**Що аналізувати:**
- **Trace tree** — видно, скільки `web_search` викликів зробив researcher, які саме запити, та яка кожного латентність
- **Model costs per span** — у Langfuse окремо для tool-calling researcher (gpt-5.4-mini) та writer (gpt-5.4), одразу видно, який агент коштував дорожче
- **Timeline** — горизонтальна шкала часу, bottleneck (найдовший span) одразу видно
- **Metadata** — фільтр за `tags:multi-agent`, `metadata.user_id:student_demo`
- **Tool inputs/outputs** — можна відкрити кожен `web_search` span і побачити запит та повернуті результати

### 2.5. Альтернатива: підключення LangSmith

Якщо замість Langfuse ви обираєте **LangSmith** — інтеграція з LangChain/LangGraph вмикається через env vars без змін у коді агентів. Trace Tree у LangSmith UI виглядатиме аналогічно.

In [8]:
# LangSmith integration (uncomment to use instead of Langfuse)
# Requires: LANGSMITH_API_KEY, LANGSMITH_PROJECT

# import os
# os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_API_KEY"] = "lsv2_..."
# os.environ["LANGSMITH_PROJECT"] = "lecture-12-observability"
#
# # No code changes needed — LangChain/LangGraph auto-detect these env vars
# # and send traces to LangSmith. Just re-run the graph.
# result = app.invoke(
#     {"topic": "Compare LangSmith and Langfuse as LLM observability platforms",
#      "research": "", "draft": ""}
# )

print("ℹ️  LangSmith code commented out — uncomment if you prefer LangSmith to Langfuse")
print("   Setup: https://smith.langchain.com → Create project → Get API key")

ℹ️  LangSmith code commented out — uncomment if you prefer LangSmith to Langfuse
   Setup: https://smith.langchain.com → Create project → Get API key


---

## 3. Управління версіями промптів (Prompt Versioning)

### 3.1. Чому хардкод промптів — це антипатерн

```python
# ❌ Антипатерн: промпт захардкоджений у коді
SYSTEM_PROMPT = """You are a helpful assistant that..."""
```

**Проблеми:**
1. **Зміна промпту = редеплой коду.** Мінімальна зміна тону → PR → CI/CD → deploy
2. **Немає історії змін.** Хто змінив промпт? Коли? Чому якість впала?
3. **Неможливість A/B тестування.** Порівняти дві версії = змінити код
4. **Блокування нетехнічних людей.** PM не може ітерувати без інженера
5. **Відсутність rollback.** Нова версія зламала якість — немає швидкого відкату

**Рішення: Prompt Registry** — промпти стають **версіонованими артефактами**:
- Кожна зміна = нова версія з metadata (автор, дата, опис)
- Environment labels: `development` → `staging` → `production`
- Rollback одним кліком
- Зв'язок з трейсами: яка версія → які результати

### Версіонування моделей

```python
# ❌ Alias — може вказувати на різні snapshots
model = "gpt-5.4"

# ✅ Конкретний snapshot — стабільна поведінка
model = "gpt-5.4-2026-03-05"

# Claude:
model = "claude-sonnet-4-6-20250514"
```

Промпти в production — це не рядки в коді, а **конфігурація системи**, яка має бути версіонованою, тестованою та відкатуваною.

### 3.2. Live-coding: Prompt Management

**Workflow з Langfuse Prompt Management:**
1. Створити промпт у UI → Prompts → Create
2. Додати template variables: `{{role}}`, `{{task}}`, `{{constraints}}`
3. Зберегти як v1, label `production`
4. Завантажити з коду через SDK

In [15]:
# Prompt Management: load a versioned prompt from Langfuse.
from langfuse import get_client

langfuse_client = get_client()

prompt = langfuse_client.get_prompt("research_agent_system", label="production")

system_message = prompt.compile(
    role="Senior Research Analyst",
    task="Analyze market trends in AI infrastructure",
    constraints="Use only verified sources, cite all claims",
)

print(f"Loaded: {prompt.name} v{prompt.version}, labels: {prompt.labels}")
print("\nCompiled system message:")
print(system_message)
print("\n✅ Change the `production` label in Langfuse UI → next run picks up the new version automatically, no redeploy.")

Loaded: research_agent_system v1, labels: ['production', 'latest']

Compiled system message:
You are a Senior Research Analyst.

Your task:
Analyze market trends in AI infrastructure
 
Constraints:
Use only verified sources, cite all claims
 
Tool usage policy: 
- You have access to a `web_search` tool. Use it to gather concrete facts, numbers, and source URLs. Prefer 2-3 focused queries over one broad query.
- Never invent sources or statistics. If a claim cannot be grounded in the search results, explicitly mark it as "unverified" or skip it.
- Stop searching once you have enough material — do not loop forever.
 
Output format:
1. A 5-7 bullet summary of the most relevant findings for the task.
2. Each bullet must end with the source URL in parentheses.
3. After the bullets, add a short `Gaps:` section listing anything you could not verify.
 
Tone: neutral, factual, concise. No marketing language, no hedging, no filler sentences.

✅ Change the `production` label in Langfuse UI → next

---

## 4. Оптимізація вартості та продуктивності

> Нагадування: **Model Routing** детально розглядали на Лекції 11. Сьогодні фокус — **кешування** та **вимірювання** ефекту через observability.

### 4.1. Рівень 1: Prompt Caching (від провайдерів)

Основні LLM-провайдери (Anthropic, OpenAI, Google) підтримують серверне кешування **префіксів** промпту: повторне звернення з тим самим початком — system prompt, tool definitions, few-shot приклади — оплачується за зниженим тарифом. Конкретні деталі (мінімальна довжина префіксу, TTL, коефіцієнти знижки, політика інвалідації) у кожного провайдера свої — перед інтеграцією варто звіряти з офіційною документацією.

#### Приклад: OpenAI

Згідно з [документацією OpenAI Prompt Caching](https://platform.openai.com/docs/guides/prompt-caching):

```
Запит 1: [System Prompt 2000 tok] + [User Msg] → повна ціна за всі input tokens
Запит 2: [System Prompt 2000 tok] + [User Msg] → cached tokens за зниженим тарифом
```

**Важливі нюанси для OpenAI:**
- Кешування **повністю автоматичне** — жодних API-ключів на кшталт `cache_control` не потрібно: просто повторно надсилайте однакові префікси
- Мінімальна довжина префіксу для кешування — **1024 токени**
- Кешується тільки **prefix** — система визначає спільний префікс автоматично
- TTL — порядок кількох хвилин (зазвичай 5–10 хв неактивності)
- Усю інформацію про кеш-хіти видає `response.usage.prompt_tokens_details.cached_tokens`
- **Антипатерн:** динамічний контент у system prompt ламає кеш

Anthropic Claude має власний механізм з явним `cache_control: {"type": "ephemeral"}` та вищим discount'ом на read — див. [їхню документацію](https://platform.claude.com/docs/en/build-with-claude/prompt-caching).

### 4.2. Рівень 2: Semantic Caching (application-level)

Ідея: для нового запиту обчислити embedding, знайти найближчий попередній запит у кеші через cosine similarity, і якщо подібність вища за threshold — повернути збережену відповідь замість виклику LLM. Добре працює на вузькому домені з повторюваними запитами (FAQ, support), у мультиагентних workflow з унікальним контекстом — значно гірше. Перед впровадженням перевіряйте гіпотезу про reuse на власному трафіку через observability.

### 4.3. Рівень 3: Контроль циклів
- `recursion_limit` — максимум кроків у LangGraph
- Timeout на tool calls (наприклад, 30 с)
- Budget per request (макс. токенів)
- `tenacity` / `backoff` для 429 errors

### 4.4. Live-coding: Prompt Caching (OpenAI)

In [19]:
# Prompt Caching with OpenAI
# 💰 Cost: ~$0.01 (2 API calls). OpenAI caches automatically — no special flags.

try:
    from openai import OpenAI
    client = OpenAI()

    # Long system prompt — must exceed 1024 tokens to be eligible for caching
    AGENT_SYSTEM_PROMPT = (
        "You are a senior AI infrastructure analyst specializing in LLM deployment, "
        "cost optimization, and production systems. You have deep expertise in: "
        "semantic caching strategies, model routing architectures, prompt engineering "
        "best practices, vector databases (Milvus, FAISS, Pinecone, Redis), "
        "LLM observability platforms (LangSmith, Langfuse, Arize), "
        "cloud deployment (AWS, GCP, Azure), containerization (Docker, Kubernetes), "
        "and API gateway patterns for LLM workloads. "
        "When answering, provide: 1) Clear explanation 2) Concrete numbers "
        "3) Code examples 4) Cost implications 5) Common pitfalls. "
        "Respond in 200-300 words. "
    ) * 10  # Repeat to exceed the 1024-token cache minimum

    approx_tokens = len(AGENT_SYSTEM_PROMPT) // 4  # rough heuristic: 1 token ≈ 4 chars
    print(f"System prompt: ~{len(AGENT_SYSTEM_PROMPT.split())} words, ~{approx_tokens} tokens\n")

    # `prompt_cache_key` is optional but strongly recommended: it tells OpenAI's
    # router to pin requests with the same prefix to the same backend replica,
    # which dramatically improves cache hit rate. Without it, the second call
    # may land on a different replica whose cache is cold.
    CACHE_KEY = "lecture-12-prompt-caching-demo"

    def run_and_report(user_msg: str):
        resp = client.chat.completions.create(
            model="gpt-4.1",
            messages=[
                {"role": "system", "content": AGENT_SYSTEM_PROMPT},
                {"role": "user", "content": user_msg},
            ],
            prompt_cache_key=CACHE_KEY,
        )
        usage = resp.usage
        cached = getattr(usage.prompt_tokens_details, "cached_tokens", 0) or 0
        print(f"  Prompt tokens : {usage.prompt_tokens}")
        print(f"  Cached tokens : {cached}")
        return cached

    # Request 1: first time the prefix is seen — no cached tokens expected
    print("📝 Request 1 (populates the cache)")
    run_and_report("What is semantic caching?")

    # Request 2: same system prefix — OpenAI serves it from cache automatically
    print("\n📝 Request 2 (should hit the cache)")
    cached_tokens = run_and_report("What is semantic caching?")

    if cached_tokens > 0:
        print(f"\n💰 {cached_tokens} tokens served from cache — billed at a discounted rate.")
    else:
        print("\nℹ️  No cache hit reported yet. Try again in a few seconds, or verify that")
        print("   the system prompt is ≥1024 tokens and identical across both calls.")

except Exception as e:
    print(f"⚠️  Skipped ({type(e).__name__}): {e}")
    print("   See the explanation above for how prompt caching works")

System prompt: ~800 words, ~1542 tokens

📝 Request 1 (populates the cache)
  Prompt tokens : 1357
  Cached tokens : 1024

📝 Request 2 (should hit the cache)
  Prompt tokens : 1357
  Cached tokens : 1024

💰 1024 tokens served from cache — billed at a discounted rate.


**Кумулятивний ефект оптимізацій вартості** _(якісна ілюстрація — довжини «стовпчиків» не є виміряними відсотками, реальний ефект залежить від профілю трафіку)_

```
Без оптимізації (один frontier model)   ██████████
Model Routing (tiered models)           ██████
+ Prompt Caching                        ████
+ Semantic Cache                        ██
                                        └─────────▶ зменшення вартості
```

Конкретні числа для вашого workload треба вимірювати через observability — див. [Langfuse Cost Tracking](https://langfuse.com/docs/observability/features/token-and-cost-tracking) та [Redis: What is Semantic Caching?](https://redis.io/blog/what-is-semantic-caching/).

---

## 5. Збір зворотного зв'язку (User Feedback)

| Тип | Приклад | Плюси | Мінуси |
|-----|---------|-------|--------|
| **Explicit** | 👍/👎 кнопки, 1-5 зірок | Чіткий сигнал | Response rate зазвичай дуже низький — більшість користувачів не залишає оцінки |
| **Implicit** | Час читання, copy/paste, повторний запит | 100% покриття | Потребує інтерпретації |

### Data Flywheel

```
👍 Фідбек → Позитивні трейси → Few-Shot приклади → Кращі агенти → Більше 👍 → ...
```

**Інтеграція:** кожна відповідь має `trace_id` → фронтенд надсилає score з прив'язкою → аналіз у дашборді

In [20]:
# Attach user feedback to the trace we just produced in Section 2.3.
# In production the score is sent from the frontend (e.g. when a user clicks 👍/👎),
# together with the `trace_id` that was returned with the response.
from langfuse import get_client
from langfuse.api import ScoreDataType

langfuse = get_client()

# Numeric 1-5 rating
langfuse.create_score(
    trace_id=last_trace_id,
    name="user_rating",
    value=4,
    data_type=ScoreDataType.NUMERIC,
    comment="Useful but could cite more sources",
)

# Boolean thumbs-up / thumbs-down
langfuse.create_score(
    trace_id=last_trace_id,
    name="thumbs_up",
    value=True,
    data_type=ScoreDataType.BOOLEAN,
)

langfuse.flush()

---

## 📌 Підсумки

1. **Observability — фундамент.** Без трейсингу ви не знаєте, чому MAS прийняла рішення. LangSmith або Langfuse — обирайте під екосистему
2. **Промпти — це конфігурація, не код.** Prompt Registry, версіонування, rollback
3. **Кешування — два рівні.** Prompt caching на стороні провайдера (для стабільних префіксів) + Semantic caching на application-level (для повторюваних запитів)
4. **Фіксуйте версії всього.** Модель (`gpt-5.4-2026-03-05`), промпт (v3, label: production), конфіг
5. **User feedback замикає цикл.** Фідбек → трейси → аналіз → покращення

---

## 📚 Ресурси

### Observability
- [Langfuse Observability](https://langfuse.com/docs/observability/overview)
- [LangSmith Observability](https://docs.langchain.com/langsmith/observability)
- [Trace LangGraph with LangSmith](https://docs.langchain.com/langsmith/trace-with-langgraph)
- [LangSmith Insights Agent](https://blog.langchain.com/insights-agent-multiturn-evals-langsmith/)
- [ClickHouse acquires Langfuse](https://clickhouse.com/blog/clickhouse-acquires-langfuse-open-source-llm-observability)

### Prompt Versioning
- [What is Prompt Management? (LangWatch)](https://langwatch.ai/blog/what-is-prompt-management-and-how-to-version-control-deploy-prompts-in-productions)
- [Top 5 Prompt Versioning Platforms 2026](https://www.getmaxim.ai/articles/top-5-prompt-versioning-platforms-in-2026/)

### Prompt Caching
- [Anthropic Prompt Caching](https://platform.claude.com/docs/en/build-with-claude/prompt-caching)
- [OpenAI Prompt Caching](https://platform.openai.com/docs/guides/prompt-caching)
- [Prompt Caching Guide (DigitalOcean)](https://www.digitalocean.com/blog/prompt-caching-with-digital-ocean)